##### ARTI 560 - Computer Vision

## Image Classification with Vision Transformer (ViT) - Exercise 

### Objective

In this exercise, you will test the pretrained Vision Transformer (ViT) model on 5 real-world images that you find online.

You will:

1. Download 5 images for different classes in [ImageNet](https://github.com/Waikato/wekaDeeplearning4j/blob/master/docs/user-guide/class-maps/IMAGENET.md).

2. Load the ImageNet class names from a [text file](https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt).

3. Use ViT to predict the class for each image.

4. Record whether the prediction was correct.

#### Important Note

For this exercise, you MUST use the following KerasHub components:

- [keras_hub.models.ViTImageClassifier](https://keras.io/keras_hub/api/models/vit/vit_image_classifier/)

- [keras_hub.models.ViTImageClassifierPreprocessor](https://keras.io/keras_hub/api/models/vit/vit_image_classifier_preprocessor/)

This ensures your input preprocessing (resizing + normalization) matches what the pretrained ViT model expects.

Do not replace the preprocessor with manual normalization (such as dividing by 255), because it may produce incorrect predictions.

In [ ]:
# Import Libraries 
# Load ViTImageClassifierPreprocessor (vit_base_patch16_224_imagenet preset) 
# Load ViTImageClassifier (vit_base_patch16_224_imagenet preset) 
# Load the images 
# Predict classes

## Import Libraries

In [1]:

import os
import csv
import urllib.request
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import tensorflow as tf

# KerasHub (must be used for ViT + Preprocessor)
import keras_hub

## Load ImageNet class names

In [2]:
IMAGENET_CLASSES_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"

def load_imagenet_class_names(url: str = IMAGENET_CLASSES_URL) -> List[str]:
    with urllib.request.urlopen(url) as resp:
        text = resp.read().decode("utf-8")
    names = [line.strip() for line in text.splitlines() if line.strip()]
    if len(names) != 1000:
        raise ValueError(f"Expected 1000 classes, got {len(names)}")
    return names

class_names = load_imagenet_class_names()
print("Loaded ImageNet class names:", len(class_names))



Loaded ImageNet class names: 1000


## Load ViT preprocessor + model

In [3]:
PRESET = "vit_base_patch16_224_imagenet"

preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(PRESET)
model = keras_hub.models.ViTImageClassifier.from_preset(PRESET)

print("Loaded preset:", PRESET)


Loaded preset: vit_base_patch16_224_imagenet


## Define 5 images  + expected labels

In [4]:
from dataclasses import dataclass
from typing import List

@dataclass(frozen=True)
class Sample:
    url: str
    expected_label: str

samples: List[Sample] = [
    Sample(
        url="https://images.unsplash.com/photo-1517849845537-4d257902454a?w=800",
        expected_label="pug",
    ),
    Sample(
        url="https://images.unsplash.com/photo-1519052537078-e6302a4968d4?w=800",
        expected_label="tiger cat",
    ),
    Sample(
        url="https://images.unsplash.com/photo-1570125909232-eb263c188f7e?w=800",
        expected_label="school bus",
    ),
    Sample(
        url="https://images.unsplash.com/photo-1603833665858-e61d17a86224?w=800",
        expected_label="banana",
    ),
    Sample(
        url="https://images.unsplash.com/photo-1552728089-57bdde30beb3?w=800",
        expected_label="lorikeet",
    ),
]

print("Samples:", len(samples))

Samples: 5


## Download & decode images 

In [5]:
def download_bytes(url: str, timeout: int = 30) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()

def decode_image(image_bytes: bytes) -> tf.Tensor:
    """
    Returns uint8 tensor [H, W, 3]. Keep uint8; preprocessor handles resizing+normalization.
    """
    img = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    if img.dtype != tf.uint8:
        img = tf.cast(img, tf.uint8)
    return img

decoded_images: List[tf.Tensor] = []
for i, s in enumerate(samples, start=1):
    b = download_bytes(s.url)
    img = decode_image(b)
    decoded_images.append(img)
    print(f"[{i}] decoded image shape={img.shape}, dtype={img.dtype}")



[1] decoded image shape=(1067, 800, 3), dtype=<dtype: 'uint8'>
[2] decoded image shape=(533, 800, 3), dtype=<dtype: 'uint8'>
[3] decoded image shape=(533, 800, 3), dtype=<dtype: 'uint8'>
[4] decoded image shape=(1422, 800, 3), dtype=<dtype: 'uint8'>
[5] decoded image shape=(1092, 800, 3), dtype=<dtype: 'uint8'>


## Predict classes with ViT

In [6]:
def predict_one(image_uint8: tf.Tensor) -> Tuple[int, float]:
    """
    Returns (predicted_index, confidence).
    """
    x = preprocessor(image_uint8)            # preprocessor must be used
    x = tf.expand_dims(x, axis=0)            # add batch dim
    logits = model(x, training=False)
    probs = tf.nn.softmax(logits, axis=-1)[0]
    idx = int(tf.argmax(probs).numpy())
    conf = float(probs[idx].numpy())
    return idx, conf

def normalize_label(s: str) -> str:
    return " ".join(s.strip().lower().split())

def is_correct(pred_name: str, expected_label: str) -> bool:
    """
    ImageNet labels sometimes have multiple synonyms separated by comma.
    We accept match if predicted name matches ANY expected synonym OR vice-versa.
    """
    pred = normalize_label(pred_name)
    expected_syns = [normalize_label(x) for x in expected_label.split(",")]
    return (pred in expected_syns) or (normalize_label(expected_label) in pred)

results = []
for i, (s, img) in enumerate(zip(samples, decoded_images), start=1):
    pred_idx, conf = predict_one(img)
    pred_name = class_names[pred_idx]
    ok = is_correct(pred_name, s.expected_label)

    results.append({
        "i": i,
        "url": s.url,
        "expected_label": s.expected_label,
        "predicted_label": pred_name,
        "predicted_index": pred_idx,
        "confidence": conf,
        "correct": ok,
    })

    print(f"\n[{i}]")
    print("Expected :", s.expected_label)
    print("Predicted:", pred_name, f"(idx={pred_idx}, conf={conf:.4f})")
    print("Correct? :", ok)


[1]
Expected : pug
Predicted: pug (idx=254, conf=0.9618)
Correct? : True

[2]
Expected : tiger cat
Predicted: tiger cat (idx=282, conf=0.5412)
Correct? : True

[3]
Expected : school bus
Predicted: passenger car (idx=705, conf=0.9730)
Correct? : False

[4]
Expected : banana
Predicted: banana (idx=954, conf=0.9971)
Correct? : True

[5]
Expected : lorikeet
Predicted: lorikeet (idx=90, conf=0.5880)
Correct? : True


## Record Your Results (table ()

In [7]:
def print_table(rows: List[dict]) -> None:
    cols = ["i", "expected_label", "predicted_label", "confidence", "correct"]
    widths = {c: max(len(c), max(len(str(r[c])) for r in rows)) for c in cols}
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print("\n" + header)
    print(sep)
    for r in rows:
        line = " | ".join(str(r[c]).ljust(widths[c]) for c in cols)
        print(line)

print_table(results)




i | expected_label | predicted_label | confidence         | correct
--+----------------+-----------------+--------------------+--------
1 | pug            | pug             | 0.9618404507637024 | True   
2 | tiger cat      | tiger cat       | 0.5411738157272339 | True   
3 | school bus     | passenger car   | 0.9729861617088318 | False  
4 | banana         | banana          | 0.9971470236778259 | True   
5 | lorikeet       | lorikeet        | 0.5879687070846558 | True   
